# 35 — Optuna Tuning and Evaluation for Regression (Objective 3)

**Objective.** Tune only the model family selected by PyCaret in the screening step, evaluate it on the untouched test set, and produce shipment-weight recommendations.

**Input.** `regression_model_input.csv`; `model_features.csv`; `pycaret_regression_selected_model.csv`.

**Output.** Tuned model, Optuna trials, final test metrics, residual plots, feature importance, warehouse-level recommendations and zone comparisons.

CatBoost was selected in the model screening step as the top-ranked family. This notebook tunes its hyperparameters with Optuna, fits the final model, and evaluates it on the held-out test set.

## 0. Setup

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "common.py").exists())
sys.path.insert(0, str(ROOT))
from src.common import *

set_style()

In [ ]:
import joblib
import numpy as np
import optuna
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor
from scipy import stats
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Load the selected model family and the fixed split

This section verifies that the model family chosen in the screening step is applied correctly and that the Objective 3 test set remains held back until final evaluation.

In [ ]:
p = obj_paths(3)
target = "product_wg_ton"

selected = pd.read_csv(p["train_eval"] / "pycaret_regression_selected_model.csv")
print(selected.to_string(index=False))
assert selected.loc[0, "selected_model_id"] == "catboost"

data = pd.read_csv(p["processed"] / "regression_model_input.csv")
features = pd.read_csv(p["feature_engine"] / "model_features.csv")["feature"].tolist()

train = data.loc[data["split"].eq("train")].copy()
test = data.loc[data["split"].eq("test")].copy()

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

print("train shape:", X_train.shape)
print("test shape:", X_test.shape)
print("selected model family:", selected.loc[0, "selected_model_name"])

> **Interpretation.**
>
> - This notebook works only on CatBoost, the model family selected in the screening step.
> - The 5,000 test rows are first used in the evaluation section below.
> - Tuning stays small: it adjusts core CatBoost settings rather than building a larger modelling system.

## 2. Optuna tuning on CatBoost

**Rule fixed before the run.**

- Optimise 5-fold training R².
- Tune only four readable hyperparameters: iterations, depth, learning rate and L2 regularisation.
- Use 15 trials to keep the search small and suitable for a BBA project.
- Do not create separate models for each zone.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def objective(trial):
    model = CatBoostRegressor(
        iterations=trial.suggest_int("iterations", 200, 450, step=50),
        depth=trial.suggest_int("depth", 4, 7),
        learning_rate=trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
    )
    scores = cross_val_score(model, X_train, y_train, scoring="r2", cv=cv, n_jobs=1)
    return scores.mean()

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=15, show_progress_bar=False)

trials = study.trials_dataframe()
trials = trials.sort_values("value", ascending=False).reset_index(drop=True)
print("best CV R2:", round(study.best_value, 6))
print("best params:", study.best_params)
print(trials[["number", "value", "params_iterations", "params_depth", "params_learning_rate", "params_l2_leaf_reg", "state"]].head(10).to_string(index=False))

save_table(trials, p["train_eval"] / "optuna_regression_trials.csv", index=False)
summary = pd.DataFrame([
    {
        "model": "CatBoostRegressor",
        "selected_by": "PyCaret screening",
        "tuned_by": "Optuna tuning",
        "best_cv_r2": study.best_value,
        **study.best_params,
        "n_trials": len(trials),
    }
])
save_table(summary, p["train_eval"] / "regression_selected_model_summary.csv", index=False)

> **Interpretation.**
>
> - Optuna ran 15 trials and the best 5-fold training R² is **0.9945**, a small but consistent improvement over the PyCaret screening result of 0.9943.
> - The top four trials share the same rounded R², showing that the hyperparameter landscape is flat near the optimum: depth 5–6 and moderate learning rates all work equally well.
> - The low variance across trials suggests the tuned settings are stable rather than a lucky single draw.

> **Decision — use the Optuna-tuned CatBoost model.**
>
> - Best 5-fold training R²: **0.9945**.
> - Best parameters: 400 iterations, depth 6, learning rate **0.0405**, L2 leaf regularisation **1.7976**.
> - The tuning step improves model settings without changing the simple project structure.
> - The next step is one final test-set evaluation.

## 3. Fit the final model and evaluate the test set

The final model is fitted on all 20,000 training rows using the parameters found by Optuna, then scored once on the 5,000 test rows it has never seen. This single evaluation gives the honest estimate of how well the model generalises to new warehouses.

In [ ]:
best_params = study.best_params.copy()
final_model = CatBoostRegressor(
    **best_params,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
)
final_model.fit(X_train, y_train)
pred = final_model.predict(X_test)

r2 = r2_score(y_test, pred)
n = len(y_test)
k = len(features)
adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - k - 1)

metric_summary = pd.DataFrame([
    {
        "model": "CatBoost_Optuna",
        "r2": r2,
        "adjusted_r2": adjusted_r2,
        "mae": mean_absolute_error(y_test, pred),
        "mse": mean_squared_error(y_test, pred),
        "rmse": mean_squared_error(y_test, pred) ** 0.5,
        "mape_percent": mean_absolute_percentage_error(y_test, pred) * 100,
    }
])
print(metric_summary.round(4).to_string(index=False))
save_table(metric_summary, p["train_eval"] / "test_metric_summary.csv", index=False)

joblib.dump(final_model, p["model"] / "catboost_train_model.pkl")
joblib.dump(
    {
        "model": "CatBoostRegressor",
        "selected_by": "PyCaret screening",
        "tuned_by": "Optuna tuning",
        "features": features,
        "best_params": best_params,
        "test_metrics": metric_summary.to_dict(orient="records")[0],
    },
    p["model"] / "regression_model_metadata.pkl",
)
print("saved model and metadata")

> **Interpretation.**
>
> - Test R² is **0.9947**, so the selected model explains almost all test-set variation in shipment weight.
> - MAE is **633.68 tons**, about **4.0003%** average percentage error.
> - Error is reported in tons because the business output is a recommended shipment weight, not an abstract score.

## 4. Residual checks

Residual plots check whether the model's errors are random or show systematic patterns. A good regression model should produce residuals centred near zero with no obvious trend against fitted values and an approximately normal distribution. Four standard diagnostics are shown: actual vs predicted, residuals vs fitted, a residual histogram, and a Q-Q plot.

In [ ]:
residuals = y_test - pred

residual_table = test[["Ware_house_ID", target]].copy()
residual_table["predicted_product_wg_ton"] = pred
residual_table["residual"] = residuals
save_table(residual_table, p["train_eval"] / "test_predictions.csv", index=False)

plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_test, y=pred, s=18, alpha=0.45, edgecolor=None)
low = min(y_test.min(), pred.min())
high = max(y_test.max(), pred.max())
plt.plot([low, high], [low, high], color="black", linestyle="--", linewidth=1)
plt.title("Objective 3 — actual vs predicted shipment weight")
plt.xlabel("Actual tons")
plt.ylabel("Predicted tons")
save_fig(p["train_eval"] / "actual_vs_predicted.png")
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(x=pred, y=residuals, s=18, alpha=0.45, edgecolor=None)
plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.title("Objective 3 — residuals vs fitted values")
plt.xlabel("Predicted tons")
plt.ylabel("Residual: actual - predicted")
save_fig(p["train_eval"] / "residuals_vs_fitted.png")
plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(residuals, bins=40, kde=True)
plt.title("Objective 3 — residual distribution")
plt.xlabel("Residual: actual - predicted")
save_fig(p["train_eval"] / "residual_histogram.png")
plt.show()

plt.figure(figsize=(6, 6))
stats.probplot(residuals, dist="norm", plot=plt)
plt.title("Objective 3 — residual Q-Q plot")
save_fig(p["train_eval"] / "residual_qq_plot.png")
plt.show()

> **Interpretation — all four diagnostics.**
>
> - **Actual vs predicted:** Points sit close to the diagonal, confirming the model tracks shipment weight well across the full range.
> - **Residuals vs fitted:** No obvious trend or fan shape; residuals scatter evenly around zero with no systematic over- or under-prediction at any fitted value.
> - **Residual histogram:** The distribution is roughly bell-shaped and centred near zero, consistent with well-behaved errors.
> - **Q-Q plot:** Points follow the reference line closely, supporting approximate normality of the residuals.
> - Taken together, the four diagnostics show the model meets standard regression assumptions adequately for a business project.

## 5. Feature importance

Feature importance shows which warehouse characteristics the model relies on most when predicting shipment weight. Understanding these drivers gives the business a starting point for operational discussions and builds trust that the model is not using arbitrary inputs. Both CatBoost's built-in importance and permutation importance on the test set are reported to cross-check the result.

In [ ]:
cat_importance = pd.DataFrame(
    {
        "feature": features,
        "importance": final_model.get_feature_importance(),
    }
).sort_values("importance", ascending=False)
print(cat_importance.head(15).round(4).to_string(index=False))
save_table(cat_importance, p["train_eval"] / "catboost_feature_importance.csv", index=False)

perm = permutation_importance(
    final_model,
    X_test,
    y_test,
    scoring="r2",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=1,
)
perm_table = pd.DataFrame(
    {
        "feature": features,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    }
).sort_values("importance_mean", ascending=False)
print(perm_table.head(15).round(5).to_string(index=False))
save_table(perm_table, p["train_eval"] / "permutation_importance.csv", index=False)

plt.figure(figsize=(9, 6))
top_perm = perm_table.head(15).sort_values("importance_mean")
plt.barh(top_perm["feature"], top_perm["importance_mean"])
plt.title("Objective 3 — permutation importance")
plt.xlabel("Drop in test R² when shuffled")
save_fig(p["train_eval"] / "permutation_importance.png")
plt.show()

> **Interpretation.**
>
> - Reported storage issues dominate the shipment-weight model.
> - Certificate grade, establishment year, temperature regulation and transport issues add smaller signals.
> - Location and ownership remain weak compared with current operating condition.
> - The project expectation is mostly supported, with storage issues much stronger than every other driver.

## 6. Shipment recommendations for every warehouse

Recommendations are produced by applying the tuned model to all 25,000 warehouses. The recommended shipment weight for each warehouse reflects what the model expects given that warehouse's current operating conditions. A separate all-warehouse model is fitted after the held-out test evaluation is complete, so the test-set integrity is preserved.

In [ ]:
all_X = data[features]
all_y = data[target]
all_model = CatBoostRegressor(
    **best_params,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
)
all_model.fit(all_X, all_y)
all_pred = all_model.predict(all_X)
joblib.dump(all_model, p["model"] / "catboost_all_warehouses_model.pkl")

original = load_preprocessed()[["Ware_house_ID", "zone", "WH_regional_zone"]]
recommendations = data[["Ware_house_ID", target, "split"]].merge(original, on="Ware_house_ID", how="left")
recommendations["recommended_product_wg_ton"] = all_pred
recommendations["difference_from_recorded_ton"] = recommendations["recommended_product_wg_ton"] - recommendations[target]
recommendations = recommendations[
    [
        "Ware_house_ID",
        "zone",
        "WH_regional_zone",
        target,
        "recommended_product_wg_ton",
        "difference_from_recorded_ton",
        "split",
    ]
]
save_table(recommendations, p["train_eval"] / "shipment_weight_recommendations.csv", index=False)
print(recommendations.head().round(2).to_string(index=False))

> **Interpretation.**
>
> - The recommendation file covers all 25,000 warehouses.
> - The all-warehouse model is fitted only after the held-out test evaluation is complete.
> - The recommendation is a modelled shipment-weight estimate based on current warehouse conditions, not an optimisation or allocation plan.

## 7. Zone and regional-zone comparison

Aggregating recommendations by zone and regional zone shows whether the model systematically suggests higher or lower volumes in different parts of the network. This is a reporting check, not a separate model — the zone summaries use the same tuned CatBoost predictions produced in the earlier sections.

In [ ]:
network_mean = recommendations["recommended_product_wg_ton"].mean()
zone_comparison = (
    recommendations.groupby("zone")
    .agg(
        warehouses=("Ware_house_ID", "count"),
        recorded_mean_ton=(target, "mean"),
        recommended_mean_ton=("recommended_product_wg_ton", "mean"),
        recommended_sd_ton=("recommended_product_wg_ton", "std"),
    )
    .reset_index()
)
zone_comparison["difference_from_network_mean_ton"] = zone_comparison["recommended_mean_ton"] - network_mean
zone_comparison = zone_comparison.sort_values("recommended_mean_ton", ascending=False)
print(zone_comparison.round(2).to_string(index=False))
save_table(zone_comparison, p["train_eval"] / "zone_comparison.csv", index=False)

regional_comparison = (
    recommendations.groupby("WH_regional_zone")
    .agg(
        warehouses=("Ware_house_ID", "count"),
        recorded_mean_ton=(target, "mean"),
        recommended_mean_ton=("recommended_product_wg_ton", "mean"),
        recommended_sd_ton=("recommended_product_wg_ton", "std"),
    )
    .reset_index()
)
regional_comparison["difference_from_network_mean_ton"] = regional_comparison["recommended_mean_ton"] - network_mean
regional_comparison = regional_comparison.sort_values("recommended_mean_ton", ascending=False)
print(regional_comparison.round(2).to_string(index=False))
save_table(regional_comparison, p["train_eval"] / "regional_zone_comparison.csv", index=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=zone_comparison, x="zone", y="recommended_mean_ton", color="#4C78A8")
plt.axhline(network_mean, color="black", linestyle="--", linewidth=1)
plt.title("Objective 3 — recommended mean shipment weight by zone")
plt.xlabel("Zone")
plt.ylabel("Recommended mean tons")
save_fig(p["train_eval"] / "zone_recommendation_comparison.png")
plt.show()

plt.figure(figsize=(9, 5))
sns.barplot(data=regional_comparison, x="WH_regional_zone", y="recommended_mean_ton", color="#59A14F")
plt.axhline(network_mean, color="black", linestyle="--", linewidth=1)
plt.title("Objective 3 — recommended mean shipment weight by regional zone")
plt.xlabel("Regional zone")
plt.ylabel("Recommended mean tons")
save_fig(p["train_eval"] / "regional_zone_recommendation_comparison.png")
plt.show()

> **Interpretation.**
>
> - East has the highest zone mean and South has the lowest, but the gap is small compared with the spread inside each zone.
> - Regional-zone differences are also modest beside the within-zone standard deviations.
> - Zone summaries are useful for reporting, but the model's main driver is still operating condition, especially storage issues.
> - Because the data is a single snapshot, these results show **association, not cause**.

## 8. Save outputs

In [ ]:
# All output files were written in the analysis sections above. Paths are listed here for reference.
print(f"test_predictions.csv                → {p['train_eval'] / 'test_predictions.csv'}")
print(f"shipment_weight_recommendations.csv → {p['train_eval'] / 'shipment_weight_recommendations.csv'}")
print(f"zone_comparison.csv                 → {p['train_eval'] / 'zone_comparison.csv'}")
print(f"regional_zone_comparison.csv        → {p['train_eval'] / 'regional_zone_comparison.csv'}")
print(f"catboost_train_model.pkl            → {p['model'] / 'catboost_train_model.pkl'}")
print(f"catboost_all_warehouses_model.pkl   → {p['model'] / 'catboost_all_warehouses_model.pkl'}")
print(f"regression_model_metadata.pkl       → {p['model'] / 'regression_model_metadata.pkl'}")
print(f"test_metric_summary.csv             → {p['train_eval'] / 'test_metric_summary.csv'}")
print(f"optuna_regression_trials.csv        → {p['train_eval'] / 'optuna_regression_trials.csv'}")
print(f"catboost_feature_importance.csv     → {p['train_eval'] / 'catboost_feature_importance.csv'}")
print(f"permutation_importance.csv          → {p['train_eval'] / 'permutation_importance.csv'}")

## 9. Checks

In [ ]:
import pandas as pd

_preds = pd.read_csv(p["train_eval"] / "test_predictions.csv")
assert _preds.shape[0] == 5000, f"expected 5000 rows, got {_preds.shape[0]}"
assert _preds.isnull().sum().sum() == 0, "nulls in test_predictions"

_recs = pd.read_csv(p["train_eval"] / "shipment_weight_recommendations.csv")
assert _recs.shape[0] == 25000, f"expected 25000 rows, got {_recs.shape[0]}"
assert _recs.isnull().sum().sum() == 0, "nulls in shipment_weight_recommendations"

_zone = pd.read_csv(p["train_eval"] / "zone_comparison.csv")
assert len(_zone) > 0, "zone_comparison is empty"
assert _zone.isnull().sum().sum() == 0, "nulls in zone_comparison"

import joblib
_model = joblib.load(p["model"] / "catboost_train_model.pkl")
assert hasattr(_model, "predict"), "loaded model has no predict method"

print("all checks passed")

---
## Summary

The Objective 3 regression model provides a recommended shipment weight for each of the 25,000 warehouses based on their current operating conditions. On average, predicted shipment weights are within **634 tons** of the recorded value — roughly a **4%** error — so a planner using the recommendation is unlikely to be off by more than a few percent. Warehouses in the East zone receive the highest recommended volumes on average and those in the South the lowest, though the within-zone variation is substantially larger than the zone-level difference, meaning zone alone is a poor guide to individual warehouse needs.

**Model performance on the held-out test set:**

- Test R²: **0.9947** (adjusted R²: **0.9947**)
- MAE: **633.68 tons** | MAPE: **4.00%** | RMSE: **841.52 tons**

**Modelling approach:**

- The model screening step selected CatBoost as the top-ranked family.
- Optuna tuned four hyperparameters over 15 trials; best 5-fold training R² reached **0.9945**.
- The final model was fitted on all 20,000 training rows and evaluated once on the untouched 5,000 test rows.
- Storage issues (`storage_issue_reported_l3m`) dominate the prediction; zone membership is a comparatively weak driver.

**Caveat:** The data is a single cross-sectional snapshot. All results show **association, not cause**. Increasing storage capacity or reducing reported problems may or may not change shipment volumes in practice — that requires controlled intervention.